
# Projected-Wannier Form-Factor Products at $\alpha=1$ and $\alpha=3$

This quick notebook tests ordered products of noncommuting rank-one projected-Wannier projectors.  It uses the repository class-A convention

$$
h(\mathbf k)=\sin k_x\,\sigma_x+\sin k_y\,\sigma_y+
(\alpha-\cos k_x-\cos k_y)\sigma_z,
$$

so $\alpha=1$ is in the topological regime and $\alpha=3$ is trivial.  The default trial orbitals are the two $\sigma_x$ eigenstates, matching the code's `trial_orbitals="X"` convention.

For each band $n=+$ or $-$ and label $a=(\mathbf r,\nu)$, the normalized projected-Wannier orbital is

$$
|w_a^{(n)}\rangle=
\frac{\Pi_n |\mathbf r,\tau_\nu\rangle}
{\sqrt{\langle \mathbf r,\tau_\nu|\Pi_n|\mathbf r,\tau_\nu\rangle}},
\qquad
P_a^{(n)}=|w_a^{(n)}\rangle\langle w_a^{(n)}|.
$$

The notebook samples random orderings, forms the regularized products on both bands,

$$
A_\pm(\epsilon;\gamma)=\overleftarrow{\prod_{a\in\gamma}}
\left[I+(\epsilon^{-1}-1)P_a^{(\pm)}\right],
$$

and then multiplies the band products,

$$
A_{\rm both}(\epsilon;\gamma)=A_+(\epsilon;\gamma)A_-(\epsilon;\gamma).
$$


In [ ]:

# CPU allocation. Edit CPU_LIST before running if you want to pin this notebook.
import os

CPU_LIST = ""  # examples: "0-15" or "0,2,4,6"; empty means leave unchanged
BLAS_THREADS = "1"


def _parse_cpu_list(spec):
    cpus = set()
    spec = spec.strip()
    if not spec:
        return None
    for part in spec.split(','):
        part = part.strip()
        if not part:
            continue
        if '-' in part:
            lo, hi = part.split('-', 1)
            cpus.update(range(int(lo), int(hi) + 1))
        else:
            cpus.add(int(part))
    return cpus

for var in ["OMP_NUM_THREADS", "MKL_NUM_THREADS", "OPENBLAS_NUM_THREADS", "NUMEXPR_NUM_THREADS"]:
    os.environ[var] = BLAS_THREADS

cpus = _parse_cpu_list(CPU_LIST)
if cpus is not None and hasattr(os, "sched_setaffinity"):
    os.sched_setaffinity(0, cpus)
    print(f"Pinned to CPUs: {sorted(cpus)}")
else:
    print("CPU affinity unchanged")
print({var: os.environ[var] for var in ["OMP_NUM_THREADS", "MKL_NUM_THREADS", "OPENBLAS_NUM_THREADS", "NUMEXPR_NUM_THREADS"]})


In [ ]:

from pathlib import Path
import json
import numpy as np
from numpy.linalg import eigh, svd
import matplotlib.pyplot as plt

try:
    plt.rcParams.update({
        "figure.dpi": 140,
        "savefig.dpi": 300,
        "font.size": 8,
        "axes.titlesize": 9,
        "axes.labelsize": 8,
        "legend.fontsize": 7,
    })
except Exception:
    pass

OUTDIR = Path("outputs")
OUTDIR.mkdir(exist_ok=True)



## Mechanical Algebra

No commutativity assumption is made.  For rank-one projectors,

$$
P_bP_a=\langle w_b|w_a\rangle |w_b\rangle\langle w_a|.
$$

Therefore an ordered sequence \(\gamma=(a_1,\ldots,a_M)\) gives

$$
P_{a_M}\cdots P_{a_1}
=
\left[\prod_{j=1}^{M-1}\langle w_{a_{j+1}}|w_{a_j}\rangle\right]
|w_{a_M}\rangle\langle w_{a_1}|.
$$

The literal projector product has only one nonzero singular value.  The useful full-spectrum object is the regularized product

$$
A_n(\lambda;\gamma)=\overleftarrow{\prod_{a\in\gamma}} (I+\lambda P_a^{(n)}),
\qquad \lambda=\epsilon^{-1}-1,
$$

which expands into all ordered form-factor chains,

$$
A_n
=I+\sum_{m\ge 1}\lambda^m\sum_{i_m>\cdots>i_1}
\left[\prod_{\ell=1}^{m-1}\langle w_{a_{i_{\ell+1}}}|w_{a_{i_\ell}}\rangle\right]
|w_{a_{i_m}}\rangle\langle w_{a_{i_1}}|.
$$

For exact complementary Hermitian bands, \(P_a^+P_b^-=0\).  Hence

$$
A_+A_- = A_-A_+,
$$

and the singular spectrum of \(A_{\rm both}=A_+A_-\) should be the union of the active spectra on the upper and lower band subspaces, up to numerical error.  This is a useful sanity check that the two band projectors are truly complementary.


In [ ]:

# Parameters for the quick run.
NX = 8
NY = 12
SAMPLES = 10
EPS = 0.2
RNG_SEED = 20260806
PERIODIC_X = True
PERIODIC_Y = True
TRIAL_ORBITALS = "X"  # "X", "Y", or "Z"

ALPHA_TOPO = 1.0
ALPHA_TRIV = 3.0

rng = np.random.default_rng(RNG_SEED)
print(dict(NX=NX, NY=NY, SAMPLES=SAMPLES, EPS=EPS, RNG_SEED=RNG_SEED, TRIAL_ORBITALS=TRIAL_ORBITALS))


In [ ]:

sigx = np.array([[0, 1], [1, 0]], dtype=complex)
sigy = np.array([[0, -1j], [1j, 0]], dtype=complex)
sigz = np.array([[1, 0], [0, -1]], dtype=complex)


def site_index(x, y, orb, nx=NX, ny=NY):
    return (x * ny + y) * 2 + orb


def basis_label(a, nx=NX, ny=NY):
    cell, orb = divmod(int(a), 2)
    x, y = divmod(cell, ny)
    return x, y, orb


def trial_spinors(kind=TRIAL_ORBITALS):
    kind = kind.upper()
    if kind == "X":
        return [np.array([1, 1], dtype=complex) / np.sqrt(2),
                np.array([1, -1], dtype=complex) / np.sqrt(2)]
    if kind == "Y":
        return [np.array([1, 1j], dtype=complex) / np.sqrt(2),
                np.array([1, -1j], dtype=complex) / np.sqrt(2)]
    if kind == "Z":
        return [np.array([1, 0], dtype=complex),
                np.array([0, 1], dtype=complex)]
    raise ValueError(kind)


def class_a_hamiltonian(nx, ny, alpha_profile, periodic_x=True, periodic_y=True):
    # Repo convention: onsite alpha*sigma_z, hoppings -sigma_z/2 - i sigma_{x,y}/2.
    d = 2 * nx * ny
    h = np.zeros((d, d), dtype=complex)
    tx = -0.5 * sigz - 0.5j * sigx
    ty = -0.5 * sigz - 0.5j * sigy

    for x in range(nx):
        for y in range(ny):
            sl = [site_index(x, y, o, nx, ny) for o in range(2)]
            h[np.ix_(sl, sl)] += alpha_profile[x] * sigz

            xp = x + 1
            if xp < nx or periodic_x:
                xp %= nx
                slp = [site_index(xp, y, o, nx, ny) for o in range(2)]
                h[np.ix_(slp, sl)] += tx
                h[np.ix_(sl, slp)] += tx.conj().T

            yp = y + 1
            if yp < ny or periodic_y:
                yp %= ny
                slp = [site_index(x, yp, o, nx, ny) for o in range(2)]
                h[np.ix_(slp, sl)] += ty
                h[np.ix_(sl, slp)] += ty.conj().T

    return 0.5 * (h + h.conj().T)


def alpha_profile(name, nx=NX):
    if name == "alpha_1_uniform":
        return np.full(nx, ALPHA_TOPO, dtype=float)
    if name == "alpha_3_uniform":
        return np.full(nx, ALPHA_TRIV, dtype=float)
    if name == "alpha_1_3_domain_wall":
        # Periodic x gives two interfaces between the topological strip and trivial region.
        alpha = np.full(nx, ALPHA_TRIV, dtype=float)
        alpha[nx // 4: 3 * nx // 4] = ALPHA_TOPO
        return alpha
    raise ValueError(name)


def band_projectors(h):
    evals, vecs = eigh(h)
    lower_vecs = vecs[:, evals < 0]
    upper_vecs = vecs[:, evals > 0]
    return {
        "lower": (lower_vecs, lower_vecs @ lower_vecs.conj().T),
        "upper": (upper_vecs, upper_vecs @ upper_vecs.conj().T),
    }, evals


def projected_trial_orbitals(projector, trial_kind=TRIAL_ORBITALS, tol=1e-12):
    taus = trial_spinors(trial_kind)
    cols = []
    labels = []
    norms = []
    d = projector.shape[0]
    for x in range(NX):
        for y in range(NY):
            sl = [site_index(x, y, o) for o in range(2)]
            for nu, tau in enumerate(taus):
                ket = np.zeros(d, dtype=complex)
                ket[sl] = tau
                proj = projector @ ket
                norm = float(np.real(np.vdot(ket, proj)))
                norms.append(norm)
                if norm > tol:
                    cols.append(proj / np.sqrt(norm))
                    labels.append((x, y, nu))
    return np.column_stack(cols), labels, np.asarray(norms)


def regularized_product_from_orbitals(wcols, seq, eps=EPS, return_matrix=False):
    lam = eps ** -1 - 1.0
    d = wcols.shape[0]
    a = np.eye(d, dtype=complex)
    for j in seq:
        w = wcols[:, j]
        # Left multiplication by I+lambda |w><w| without forming the dense factor.
        a += lam * np.outer(w, w.conj() @ a)
    logsv = np.log(np.maximum(svd(a, compute_uv=False), 1e-300))
    if return_matrix:
        return logsv, a
    return logsv


def active_log_singular_values(full_matrix, band_vecs):
    block = band_vecs.conj().T @ full_matrix @ band_vecs
    return np.log(np.maximum(svd(block, compute_uv=False), 1e-300))


def bare_chain_log_amplitude_phase(wcols, seq, closed=False):
    if len(seq) < 2:
        return 0.0, 0.0
    pairs = list(zip(seq[:-1], seq[1:]))
    if closed:
        pairs.append((seq[-1], seq[0]))
    log_abs = 0.0
    phase = 0.0
    for a, b in pairs:
        overlap = np.vdot(wcols[:, b], wcols[:, a])
        log_abs += np.log(max(abs(overlap), 1e-300))
        phase += np.angle(overlap)
    phase = np.angle(np.exp(1j * phase))
    return log_abs, phase


def random_sequence(num_labels, rng):
    return rng.permutation(num_labels)


In [ ]:

def run_profile(name):
    h = class_a_hamiltonian(NX, NY, alpha_profile(name), PERIODIC_X, PERIODIC_Y)
    projectors, evals = band_projectors(h)
    profile_result = {
        "evals": evals,
        "alpha": alpha_profile(name),
        "bands": {},
        "combined": [],
        "combined_commutator_norm": [],
        "combined_union_error": [],
    }

    projected = {}
    for band, (vecs, projector) in projectors.items():
        wcols, labels, norms = projected_trial_orbitals(projector)
        projected[band] = (vecs, wcols, labels, norms)
        profile_result["bands"][band] = {
            "labels": labels,
            "diag_norms": norms,
            "active_logsv": [],
            "full_logsv": [],
            "bare_open_logabs": [],
            "bare_closed_logabs": [],
            "bare_closed_phase": [],
        }

    for sample in range(SAMPLES):
        # Use the same center/trial ordering for upper and lower in a sample.
        seq = random_sequence(len(projected["lower"][2]), rng)
        sample_matrices = {}
        sample_active = {}

        for band in ["lower", "upper"]:
            vecs, wcols, labels, norms = projected[band]
            full_logsv, amat = regularized_product_from_orbitals(wcols, seq, return_matrix=True)
            active_logsv = active_log_singular_values(amat, vecs)
            open_log, _ = bare_chain_log_amplitude_phase(wcols, seq, closed=False)
            closed_log, closed_phase = bare_chain_log_amplitude_phase(wcols, seq, closed=True)

            profile_result["bands"][band]["full_logsv"].append(full_logsv)
            profile_result["bands"][band]["active_logsv"].append(active_logsv)
            profile_result["bands"][band]["bare_open_logabs"].append(open_log)
            profile_result["bands"][band]["bare_closed_logabs"].append(closed_log)
            profile_result["bands"][band]["bare_closed_phase"].append(closed_phase)
            sample_matrices[band] = amat
            sample_active[band] = active_logsv

        combined_ul = sample_matrices["upper"] @ sample_matrices["lower"]
        combined_lu = sample_matrices["lower"] @ sample_matrices["upper"]
        combined_logsv = np.log(np.maximum(svd(combined_ul, compute_uv=False), 1e-300))
        union_logsv = np.sort(np.concatenate([sample_active["upper"], sample_active["lower"]]))[::-1]
        profile_result["combined"].append(combined_logsv)
        profile_result["combined_union_error"].append(float(np.max(np.abs(combined_logsv - union_logsv))))
        profile_result["combined_commutator_norm"].append(float(np.linalg.norm(combined_ul - combined_lu) / max(np.linalg.norm(combined_ul), 1e-300)))

    for band in ["lower", "upper"]:
        for key in ["active_logsv", "full_logsv", "bare_open_logabs", "bare_closed_logabs", "bare_closed_phase"]:
            profile_result["bands"][band][key] = np.asarray(profile_result["bands"][band][key])
    profile_result["combined"] = np.asarray(profile_result["combined"])
    profile_result["combined_union_error"] = np.asarray(profile_result["combined_union_error"])
    profile_result["combined_commutator_norm"] = np.asarray(profile_result["combined_commutator_norm"])
    return profile_result

profiles = ["alpha_1_uniform", "alpha_3_uniform", "alpha_1_3_domain_wall"]
pretty = {
    "alpha_1_uniform": r"uniform $\alpha=1$",
    "alpha_3_uniform": r"uniform $\alpha=3$",
    "alpha_1_3_domain_wall": r"domain wall $1|3$",
}
results = {name: run_profile(name) for name in profiles}

summary = {}
for name, res in results.items():
    summary[name] = {
        "gap_min_abs": float(np.min(np.abs(res["evals"]))),
        "negative_states": int(np.sum(res["evals"] < 0)),
        "positive_states": int(np.sum(res["evals"] > 0)),
        "both_band_union_error_max": float(np.max(res["combined_union_error"])),
        "both_band_commutator_norm_max": float(np.max(res["combined_commutator_norm"])),
    }
summary


In [ ]:

# Save a compact cache for later reuse.
cache_path = OUTDIR / "projector_form_factor_random_sequences_results.npz"
np.savez_compressed(
    cache_path,
    profiles=np.array(profiles),
    params=json.dumps({
        "NX": NX,
        "NY": NY,
        "SAMPLES": SAMPLES,
        "EPS": EPS,
        "RNG_SEED": RNG_SEED,
        "ALPHA_TOPO": ALPHA_TOPO,
        "ALPHA_TRIV": ALPHA_TRIV,
        "TRIAL_ORBITALS": TRIAL_ORBITALS,
        "PERIODIC_X": PERIODIC_X,
        "PERIODIC_Y": PERIODIC_Y,
    }),
    **{f"{name}_alpha": res["alpha"] for name, res in results.items()},
    **{f"{name}_evals": res["evals"] for name, res in results.items()},
    **{f"{name}_{band}_active_logsv": res["bands"][band]["active_logsv"] for name, res in results.items() for band in ["lower", "upper"]},
    **{f"{name}_{band}_full_logsv": res["bands"][band]["full_logsv"] for name, res in results.items() for band in ["lower", "upper"]},
    **{f"{name}_{band}_bare_open_logabs": res["bands"][band]["bare_open_logabs"] for name, res in results.items() for band in ["lower", "upper"]},
    **{f"{name}_{band}_bare_closed_logabs": res["bands"][band]["bare_closed_logabs"] for name, res in results.items() for band in ["lower", "upper"]},
    **{f"{name}_{band}_bare_closed_phase": res["bands"][band]["bare_closed_phase"] for name, res in results.items() for band in ["lower", "upper"]},
    **{f"{name}_both_band_logsv": res["combined"] for name, res in results.items()},
    **{f"{name}_both_band_union_error": res["combined_union_error"] for name, res in results.items()},
    **{f"{name}_both_band_commutator_norm": res["combined_commutator_norm"] for name, res in results.items()},
)
print(cache_path)
print(json.dumps(summary, indent=2))


In [ ]:

# Figure 1: alpha profiles and single-particle spectra.
fig, axs = plt.subplots(1, 2, figsize=(6.75, 2.4), constrained_layout=True)
for name in profiles:
    axs[0].plot(np.arange(NX), results[name]["alpha"], marker="o", label=pretty[name])
axs[0].set_xlabel("x")
axs[0].set_ylabel(r"$\alpha(x)$")
axs[0].set_title("Alpha profiles")
axs[0].legend(frameon=False)

for name in profiles:
    evals = results[name]["evals"]
    axs[1].plot(np.arange(len(evals)), evals, marker=".", linestyle="", ms=2, label=pretty[name])
axs[1].axhline(0, color="k", lw=0.7)
axs[1].set_xlabel("eigenvalue index")
axs[1].set_ylabel("single-particle energy")
axs[1].set_title("Domain wall produces near-zero modes")
axs[1].legend(frameon=False)
fig.savefig(OUTDIR / "fig1_alpha_profiles_and_energy_spectra.png", bbox_inches="tight")
plt.show()


In [ ]:

# Figure 2: active regularized-product log-singular-value spectra for lower/upper bands.
colors = {"alpha_1_uniform": "tab:orange", "alpha_3_uniform": "tab:blue", "alpha_1_3_domain_wall": "tab:red"}
fig, axs = plt.subplots(1, 2, figsize=(6.75, 2.6), sharey=True, constrained_layout=True)
for ax, band in zip(axs, ["lower", "upper"]):
    for name in profiles:
        data = results[name]["bands"][band]["active_logsv"]
        mean = data.mean(axis=0)
        err = data.std(axis=0, ddof=1)
        x = np.arange(1, len(mean) + 1)
        ax.plot(x, mean, color=colors[name], lw=1.2, label=pretty[name])
        ax.fill_between(x, mean - err, mean + err, color=colors[name], alpha=0.14, lw=0)
    ax.set_title(f"{band} band product")
    ax.set_xlabel("singular-value rank")
    ax.grid(alpha=0.2)
axs[0].set_ylabel(r"$\log s_j[A_n(\epsilon)]$")
axs[1].legend(frameon=False, loc="upper right")
fig.savefig(OUTDIR / "fig2_active_band_log_singular_spectra_alpha1_alpha3.png", bbox_inches="tight")
plt.show()


In [ ]:

# Figure 3: both-band multiplied product A_+(epsilon) A_-(epsilon), requested alpha=1 and alpha=3 comparison.
fig, ax = plt.subplots(figsize=(3.375, 2.55), constrained_layout=True)
for name in profiles:
    data = results[name]["combined"]
    mean = data.mean(axis=0)
    err = data.std(axis=0, ddof=1)
    x = np.arange(1, len(mean) + 1)
    ax.plot(x, mean, lw=1.15, color=colors[name], label=pretty[name])
    ax.fill_between(x, mean - err, mean + err, color=colors[name], alpha=0.12, lw=0)
ax.set_xlabel("singular-value rank")
ax.set_ylabel(r"$\log s_j[A_+(\epsilon)A_-(\epsilon)]$")
ax.set_title("both-band product")
ax.grid(alpha=0.2)
ax.legend(frameon=False)
fig.savefig(OUTDIR / "fig3_both_band_product_alpha1_alpha3_domain_wall.png", bbox_inches="tight")
plt.show()


In [ ]:

# Figure 4: domain-wall excess of the both-band product relative to the average of uniform alpha=1 and alpha=3.
dw = results["alpha_1_3_domain_wall"]["combined"].mean(axis=0)
a1 = results["alpha_1_uniform"]["combined"].mean(axis=0)
a3 = results["alpha_3_uniform"]["combined"].mean(axis=0)
bulk_avg = 0.5 * (a1 + a3)

fig, ax = plt.subplots(figsize=(3.375, 2.45), constrained_layout=True)
ax.plot(np.arange(1, len(dw) + 1), dw - bulk_avg, color="tab:red", lw=1.2)
ax.axhline(0, color="k", lw=0.7)
ax.set_title("both-band domain-wall excess")
ax.set_xlabel("singular-value rank")
ax.set_ylabel(r"$\Delta \log s_j$")
ax.grid(alpha=0.2)
fig.savefig(OUTDIR / "fig4_both_band_domain_wall_excess.png", bbox_inches="tight")
plt.show()


In [ ]:

# Figure 5: literal bare projector chains collapse to one singular value; this plots its log amplitude.
fig, axs = plt.subplots(1, 2, figsize=(6.75, 2.55), constrained_layout=True)
width = 0.22
x0 = np.arange(len(profiles))
for k, band in enumerate(["lower", "upper"]):
    means = np.array([results[name]["bands"][band]["bare_open_logabs"].mean() for name in profiles])
    errs = np.array([results[name]["bands"][band]["bare_open_logabs"].std(ddof=1) for name in profiles])
    axs[0].bar(x0 + (k - 0.5) * width, means, width=width, yerr=errs, capsize=2, label=band)
axs[0].set_xticks(x0, [pretty[name] for name in profiles], rotation=25, ha="right")
axs[0].set_ylabel(r"$\log |\prod_j \langle w_{a_{j+1}}|w_{a_j}\rangle|$")
axs[0].set_title("open rank-one chain")
axs[0].legend(frameon=False)

for name in profiles:
    phase = results[name]["bands"]["lower"]["bare_closed_phase"]
    amp = results[name]["bands"]["lower"]["bare_closed_logabs"]
    axs[1].scatter(phase, amp, s=18, alpha=0.75, label=pretty[name])
axs[1].set_xlabel("closed-chain phase")
axs[1].set_ylabel("closed-chain log amplitude")
axs[1].set_title("lower-band Bargmann loops")
axs[1].legend(frameon=False)
fig.savefig(OUTDIR / "fig5_bare_bargmann_chain_amplitudes_alpha1_alpha3.png", bbox_inches="tight")
plt.show()


In [ ]:

# Figure 6: complementary-band union and commutator checks for A_+ A_-.
fig, axs = plt.subplots(1, 2, figsize=(6.75, 2.55), constrained_layout=True)
errs = [results[name]["combined_union_error"] for name in profiles]
axs[0].boxplot(errs, labels=[pretty[name] for name in profiles], showfliers=False)
axs[0].set_yscale("log")
axs[0].set_ylabel(r"max $|\log s(A_+A_-)-\log s(A_+)\cup\log s(A_-)|$")
axs[0].set_title("active-spectrum union check")
axs[0].tick_params(axis="x", rotation=25)

comm = [results[name]["combined_commutator_norm"] for name in profiles]
axs[1].boxplot(comm, labels=[pretty[name] for name in profiles], showfliers=False)
axs[1].set_yscale("log")
axs[1].set_ylabel(r"$\|A_+A_--A_-A_+\|/\|A_+A_-\|$")
axs[1].set_title("both-band commutator check")
axs[1].tick_params(axis="x", rotation=25)
fig.savefig(OUTDIR / "fig6_both_band_union_and_commutator_checks.png", bbox_inches="tight")
plt.show()



## What This Quick Test Suggests

1. The bare product is not a many-direction transfer object.  It is a rank-one ordered Bargmann chain.  Its only nonzero singular value is the modulus of an ordered product of form factors, and its phase is a discrete Berry/Bargmann phase for a closed chain.

2. The regularized product is the useful single-particle transfer object.  It keeps all ordered subsequences of form-factor chains and has a full log-singular-value spectrum.

3. Multiplying both bands is straightforward for exact Hermitian complementary projectors: \(A_+A_-\) acts blockwise on the upper/lower band subspaces, and its active singular spectrum is the union of the two band spectra.  The notebook checks both the union property and the small commutator.

4. The domain-wall profile changes the spectrum because the sign-band projector contains near-zero interface modes.  Those modes change the real-space form-factor kernel, especially along the wall, so random ordered products tend to have enhanced leading log-singular values relative to the uniform \(\alpha=1\) and \(\alpha=3\) controls.

5. This is not yet a topological invariant.  The result depends on ordering, regulator, finite size, and whether the sequence samples the wall.  A more invariant next step would compare closed-loop phases under controlled paths around the interface, or scale the domain-wall excess with \(N_y\) while separating wall-centered and bulk-centered projected orbitals.
